# Case study: first SQL questions about the BTI decisions

*Session 3, block 1 · Author: course team · Licence: CC-BY-4.0*

In this notebook you answer first questions about the course data in SQL. We use **DuckDB**, a database
that runs inside the Python process and can query Parquet files directly. Nothing needs to be installed
beyond the course environment, and the SQL is the same as in PostgreSQL for everything we use here.

The data: `decisions` holds 309,529 Binding Tariff Information (BTI) decisions that started in 2017–2023;
`nomenclature` holds the 1,229 four-digit headings of the Harmonized System (HS 2022) with English
descriptions, chapters and sections.

**Tasks of the practice block**

1. First queries: size, keys, filters.
2. Decisions per issuing country and year (GROUP BY, HAVING).
3. Join the headings to their English names (JOIN).
4. Headings without any decision, and decisions without a known heading (LEFT JOIN, NULL).

Cells marked **TODO** are for you. Solutions are not included; compare your numbers with your neighbours.

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data"
assert (DATA / "train.parquet").exists(), "run case-study/prepare_data.py first"

import duckdb

con = duckdb.connect()  # an in-memory database
# views: names for the Parquet files, so that we can write FROM decisions
con.execute(f"CREATE VIEW decisions    AS SELECT * FROM read_parquet('{DATA / 'train.parquet'}')")
con.execute(f"CREATE VIEW nomenclature AS SELECT * FROM read_parquet('{DATA / 'nomenclature.parquet'}')")


def q(sql: str):
    """Run a query and return the result as a pandas DataFrame."""
    return con.sql(sql).df()


q("SELECT COUNT(*) AS n_decisions FROM decisions")

## 1. Look at the tables

`DESCRIBE` lists the columns and their types. A **primary key** identifies each row: `bti_reference` in
`decisions`, `heading` in `nomenclature`. `decisions.heading` is a **foreign key**: it refers to a row of
`nomenclature`.

In [ ]:
q("DESCRIBE decisions")

In [ ]:
q("DESCRIBE nomenclature")

In [ ]:
# is bti_reference really unique? A primary key must be.
q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT bti_reference) AS n_ids FROM decisions")

## 2. Filtering and sorting: SELECT, WHERE, ORDER BY, LIMIT

Most descriptions are in German or French. The English `keywords`, assigned by customs, help you read
them. (Keywords are decided together with the classification; later sessions never use them as model
input.)

In [ ]:
q("""
SELECT bti_reference, issuing_country, start_date::date AS start_date, LEFT(keywords, 50) AS keywords
FROM decisions
WHERE heading = '9503' AND language = 'en'     -- toys, described in English
ORDER BY start_date DESC, bti_reference        -- newest first; the reference breaks ties
LIMIT 5
""")

In [ ]:
# how are the descriptions distributed over languages?
q("""
SELECT language, COUNT(*) AS n, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM decisions
GROUP BY language
ORDER BY n DESC
LIMIT 8
""")

**TODO 1.** How many decisions have the keyword *face mask* (case-insensitive, `ILIKE '%face mask%'`)?
Under which headings were they classified? How many decisions of chapter 63 started in 2020?

In [ ]:
# TODO 1: replace ... by a condition, then remove the comment signs
# q("""
# SELECT heading, COUNT(*) AS n
# FROM decisions
# WHERE ...
# GROUP BY heading
# ORDER BY n DESC
# """)

## 3. Decisions per country and year: GROUP BY and HAVING

`WHERE` filters rows before grouping; `HAVING` filters groups after aggregation.

In [ ]:
q("""
SELECT issuing_country, COUNT(*) AS n_decisions, COUNT(DISTINCT heading) AS n_headings
FROM decisions
GROUP BY issuing_country
HAVING COUNT(*) >= 5000          -- only countries with at least 5000 decisions
ORDER BY n_decisions DESC
""")

In [ ]:
# decisions per year for the four largest issuing countries; FILTER counts only the matching rows
q("""
SELECT EXTRACT(YEAR FROM start_date)::int AS year,
       COUNT(*) AS total,
       COUNT(*) FILTER (WHERE issuing_country = 'DE') AS de,
       COUNT(*) FILTER (WHERE issuing_country = 'FR') AS fr,
       COUNT(*) FILTER (WHERE issuing_country = 'NL') AS nl,
       COUNT(*) FILTER (WHERE issuing_country = 'GB') AS gb
FROM decisions
GROUP BY year
ORDER BY year
""")

**TODO 2.** (a) The United Kingdom (`GB`) issues no decisions after 2020. Which country code appears in
2021 instead, for decisions in English? (Hint: group by `issuing_country` for `language = 'en'` and
`start_date >= '2021-01-01'`.) (b) Which issuing countries write in more than one language?

In [ ]:
# TODO 2

## 4. Headings with their English names: INNER JOIN

The English description is an attribute of the heading, so we join each decision with its row of the
nomenclature. With an inner join, decisions without a matching heading would disappear. Do they exist?
Section 5 checks.

In [ ]:
q("""
SELECT d.heading, LEFT(n.heading_description, 60) AS heading_description, COUNT(*) AS n_decisions
FROM decisions AS d
JOIN nomenclature AS n ON n.heading = d.heading
GROUP BY d.heading, n.heading_description
ORDER BY n_decisions DESC
LIMIT 10
""")

In [ ]:
# the same per section (21 sections): a coarser view of the nomenclature
q("""
SELECT n.section, LEFT(n.section_name, 50) AS section_name, COUNT(*) AS n_decisions,
       COUNT(DISTINCT d.heading) AS n_headings
FROM decisions AS d
JOIN nomenclature AS n USING (heading)
GROUP BY n.section, n.section_name
ORDER BY n_decisions DESC
""")

**TODO 3.** Which are the three most frequent headings in France (`FR`) and in Poland (`PL`), with
their English names? Are they the same as in Germany?

In [ ]:
# TODO 3

## 5. LEFT JOIN, anti-joins and NULL

`NULL` means *unknown*. Any comparison with NULL is unknown, so `keywords = NULL` is never true; use
`keywords IS NULL`. `COUNT(*)` counts rows, `COUNT(column)` counts non-NULL values.

In [ ]:
q("""
SELECT COUNT(*)                     AS n_decisions,
       COUNT(keywords)              AS n_with_keywords,
       COUNT(invalidation_reason)   AS n_with_invalidation_reason,
       COUNT(*) - COUNT(keywords)   AS n_without_keywords
FROM decisions
""")

In [ ]:
# the classic mistake: = NULL instead of IS NULL
q("SELECT (SELECT COUNT(*) FROM decisions WHERE keywords = NULL) AS eq_null, "
  "(SELECT COUNT(*) FROM decisions WHERE keywords IS NULL) AS is_null")

In [ ]:
# anti-join: headings of the nomenclature without any decision in 2017-2023
q("""
SELECT n.heading, LEFT(n.heading_description, 60) AS heading_description
FROM nomenclature AS n
LEFT JOIN decisions AS d ON d.heading = n.heading
WHERE d.bti_reference IS NULL
ORDER BY n.heading
""")

In [ ]:
# the other direction: decisions whose heading is not in the nomenclature
q("""
SELECT d.heading, COUNT(*) AS n_decisions, MIN(d.start_date)::date AS first, MAX(d.start_date)::date AS last
FROM decisions AS d
LEFT JOIN nomenclature AS n ON n.heading = d.heading
WHERE n.heading IS NULL
GROUP BY d.heading
""")

Heading 8803 (parts of aircraft) was deleted by the HS 2022 revision; such parts now belong to 8807.
The nomenclature table is the 2022 version, so an inner join silently loses these decisions. A foreign key
in PostgreSQL would have refused them at load time (block 2).

**TODO 4.** (a) In which chapters are the headings without decisions? Count them per chapter and join the
chapter description. Why are there hardly any decisions for these goods? (b) Load the test table
(`test.parquet`) as a view `decisions_test`: how many test decisions are there per issuing country? Which
columns does the test table not have, and why?

In [ ]:
# TODO 4

## 6. The same in PostgreSQL

After block 2 your data are in PostgreSQL. Every query above runs there unchanged; replace the DuckDB
connection by an SQLAlchemy engine:

```python
import os, pandas as pd
from sqlalchemy import create_engine
engine = create_engine(os.environ["DATABASE_URL"])
pd.read_sql("SELECT COUNT(*) FROM decisions", engine)
```